# 3. Streaming forecasting (RLS and ML) with Spark Structured Streaming

#### *Gianluca Bontempi, Cédric Simar, and Martin Colot - materials from Jacopo De Stefani and Theo Verhelst*

This notebook provides an example for performing machine learning analytics on streaming data produced locally.

The first part contains examples of simple __stateless__ processing scripts.

The second part introduces __stateful__ streaming processing in order to perform more complex operations, like cumulative sum, sequential estimation of mean and variance.

Predictive analytics is also implemented thanks to stateful processing.

Two main __streaming learning__ strategies are proposed:
* an online linear learning where the coefficient estimation is achieved using the __recursive least square (RLS)__ algorithm
* a __mini-batch strategy__ where a learning model is fit to most recent data

A __racing__ strategy is implemented as well where two different models (e.g. two linear models with different forgetting factors or a linear and a nonlinear regressor) are fitted to the same data.

This notebook uses:
* [Spark Structured Streaming](https://spark.apache.org/docs/latest/structured-streaming-programming-guide.html) for processing the streaming data using DataFrames
* [Spark SQL](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/index.html) for processing the dataFrames


See also :
* [Spark streaming](https://spark.apache.org/docs/latest/streaming-programming-guide.html) for processing the streaming data using RDD (legacy code, not updated since 2016 !)


## General imports

In [ ]:
import time
import re, ast
import numpy as np
import matplotlib.pyplot as plt
import sqlite3

from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn import linear_model
from sklearn.linear_model import LinearRegression

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, expr, rand, udf
from pyspark.sql.types import StringType

## Spark Streaming Session

After creating a spark session, we use `spark.readStream` to create a streaming dataFrame. This data frame loads data automatically from the given source : file, Kafka, socket or rate. The `rate` source works [as follows](https://spark.apache.org/docs/latest/structured-streaming-programming-guide.html) :

_Generates data at the specified number of rows per second, each output row contains a timestamp and value. Where timestamp is a Timestamp type containing the time of message dispatch, and value is of Long type containing the message count, starting from 0 as the first row. This source is intended for testing and benchmarking._

In [ ]:
# Initialize Spark Session (local mode)
spark = SparkSession.builder \
    .appName("StreamingAnalytics") \
    .master("local[2]") \
    .getOrCreate()

#spark.conf.set("spark.sql.shuffle.partitions", "2") # reduce number of partitions to run on google Colab

In [ ]:
# Create a streaming DataFrame
streaming_df = spark.readStream \
    .format("rate") \
    .option("rowsPerSecond", 5) \
    .load()

After that, we use `writeStream` to output the processed stream in local memory under a given query name

In [ ]:
# Create a query to output the data
query = streaming_df.writeStream \
    .outputMode("append") \
    .format("memory") \
    .queryName("streaming_data") \
    .start()

time.sleep(2)

We retreive the output from the memory using `spark.sql`

In [ ]:
spark.sql("SELECT * FROM streaming_data ORDER BY value DESC LIMIT 10").toPandas()

,timestamp,value
0,2025-03-25 09:38:25.198,24
1,2025-03-25 09:38:24.998,23
2,2025-03-25 09:38:24.798,22
3,2025-03-25 09:38:24.598,21
4,2025-03-25 09:38:24.398,20
5,2025-03-25 09:38:24.198,19
6,2025-03-25 09:38:23.998,18
7,2025-03-25 09:38:23.798,17
8,2025-03-25 09:38:23.598,16
9,2025-03-25 09:38:23.398,15


Finally, we can stop the query

In [ ]:
query.stop()

and stop the spark session if necessary

In [ ]:
# spark.stop()

## Dataframes transformations

To process the streamed data, we can use most of the common operations on DataFrames. For example, `spark.sql.DataFrame.select` allows to create new streaming dataFrames by selecting columns from another one and applying transformations.

In [ ]:
# Create a streaming DataFrame
streaming_df = spark.readStream \
    .format("rate") \
    .option("rowsPerSecond", 5) \
    .load()

In [ ]:
# Process the DataFrame
processed_df = streaming_df.select(
    col("value").alias("id"),
    (rand()*10).alias("x"),                         # generate random values
    (3 * col("x") + 7 + rand() * 2 - 1).alias("y")  # y = 3x + 7 + noise
)

In [ ]:
# Create a query to output the data
query = processed_df.writeStream \
    .outputMode("append") \
    .format("memory") \
    .queryName("streaming_data") \
    .start()

time.sleep(2)

In [ ]:
spark.sql("SELECT * FROM streaming_data ORDER BY id DESC LIMIT 10").toPandas()

,id,x,y
0,9,2.342644,14.187253
1,8,6.381064,25.216784
2,7,4.984455,22.116129
3,6,7.329596,28.430538
4,5,8.248948,32.209288
5,4,2.872950,16.366616
6,3,8.260830,32.778398
7,2,9.452181,35.684978
8,1,9.326820,34.982520
9,0,0.456423,7.440881


In [ ]:
query.stop()

### Exercise 1.

The following cell simulates the acquisition of data from multiple persons in 4 countries. Each record contains the country where the person is, the device that the person uses and the screen time of the person for a day (how much time the person spent on the device, in hours).

Use Dataframe opetations to find, for each record, if the screen time is greater than 3 hours. Then, for each pair of country and device, count the number of remaining records. Finally, order the pairs according to this count, in decreasing order.

In [ ]:
# Create a streaming DataFrame
streaming_df = spark.readStream \
    .format("rate") \
    .option("rowsPerSecond", 100) \
    .load()

devices = ["Phone", "Laptop", "Watch"]
country = ["Belgium", "France", "Germany", "Luxembourg"]

# Process the DataFrame
input_df = streaming_df.select(
    col("value").alias("id"),
    (udf(lambda x : devices[int(x * len(devices))], StringType())(rand())).alias("device"), # user defined function (UDF) to map random numbers to strings
    (udf(lambda x : country[int(x * len(country))], StringType())(rand())).alias("country"), # user defined function (UDF) to map random numbers to strings
    (rand()*6).alias("screen_time")
)

**Your solution :**

**Solution :**

In [ ]:
processed_df = input_df.select("id", "country", "device").where("screen_time > 3").groupBy("country", "device").count().sort(col("count").desc())

In [ ]:
# Create a query to output the data
# use complete mode instead of append mode because of aggregation groupBy
query = processed_df.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("streaming_data") \
    .start()

In [ ]:
spark.sql("SELECT * FROM streaming_data LIMIT 5").toPandas() # can take some time

,country,device,count
0,Germany,Watch,11
1,Luxembourg,Laptop,9
2,Germany,Laptop,9
3,France,Phone,8
4,France,Watch,8


In [ ]:
query.stop()

To update the count at each micro batch without recomuting it from scratch, Spark internally uses hidden states.

## Stateful Batch processing

We can define our own states to compute and update complex aggregations over micro bataches.
In the next example, we use custom states to compute the average of a variable and update it at each micro-match.

It is important to note that the state must be stored in an external database to make it persistent across batches when they are processed on different workers (which does not happen if you run the code on google colab but happens in real infratructures).
Here, we use a simple local sqlite database. For real world applications, refer to solutions like [Redis](https://redis.io/docs/latest/integrate/redis-py/), or [delta lake](https://delta.io/)

Note: This is a simple example of the idea behind statefull processing. However to compute the average of all records, it is sufficient to apply the following operation on DataFrames : `processed_df.agg({"value" : "mean"})`

In [ ]:
conn = sqlite3.connect('/content/state.db')
cursor = conn.cursor()

cursor.execute('''DROP TABLE IF EXISTS state''')

In [ ]:
# Function to update state in SQLite
def update_state(batch_df, batch_id):
    if batch_df.count() > 0:
        # Create a new SQLite connection inside the function
        conn = sqlite3.connect('/content/state.db')
        cursor = conn.cursor()

        # Ensure state table exists
        cursor.execute('''
            CREATE TABLE IF NOT EXISTS state (
                total REAL,
                count INTEGER
            )
        ''')

        # Calculate new batch sum and count
        total_sum = batch_df.agg({"value": "sum"}).collect()[0][0]
        count = batch_df.count()

        # Read previous state from SQLite
        cursor.execute('SELECT total, count FROM state')
        row = cursor.fetchone()
        if row:
            prev_total, prev_count = row
        else:
            prev_total, prev_count = 0, 0  # Initialize if no state exists

        # Compute new state
        new_total = prev_total + total_sum
        new_count = prev_count + count
        new_avg = new_total / new_count if new_count > 0 else 0

        # Store updated state in SQLite
        cursor.execute('DELETE FROM state')  # Remove old state (keeping only one row)
        cursor.execute('INSERT INTO state (total, count) VALUES (?, ?)', (new_total, new_count))
        conn.commit()

        # Close SQLite connection
        cursor.close()
        conn.close()

        display(f"Updated State -> total: {new_total}, count: {new_count}, average: {new_avg}")


In [ ]:
# Create a streaming DataFrame
streaming_df = spark.readStream \
    .format("rate") \
    .option("rowsPerSecond", 10) \
    .load()

# Process the DataFrame
processed_df = streaming_df.select(
    col("value").alias("id"),
    (rand()*100).alias("value")  # generate random values
)


# Start the streaming query
query = processed_df.writeStream \
    .outputMode("append") \
    .format("memory") \
    .foreachBatch(update_state) \
    .start()


time.sleep(10)

'Updated State -> total: 613.2997955711952, count: 10, average: 61.32997955711952'

'Updated State -> total: 969.1432070767469, count: 20, average: 48.457160353837345'

'Updated State -> total: 1268.0427488058303, count: 30, average: 42.268091626861015'

'Updated State -> total: 1743.2985920535202, count: 40, average: 43.582464801338006'

'Updated State -> total: 2152.70178043934, count: 50, average: 43.0540356087868'

'Updated State -> total: 2566.9834479811016, count: 60, average: 42.78305746635169'

'Updated State -> total: 3060.1855991291545, count: 70, average: 43.71693713041649'

'Updated State -> total: 3511.7794082382125, count: 80, average: 43.897242602977656'

'Updated State -> total: 3965.0588530917316, count: 90, average: 44.056209478797015'

In [ ]:
cursor.execute('SELECT total, count FROM state')
row = cursor.fetchone()
avg = 0
if row:
    total, count = row
    avg = total / count if count > 0 else 0
print(f"total : {total}, count : {count}, average : {avg}")

total : 8958.494994376591, count : 200, average : 44.79247497188296


In [ ]:
query.stop()

### Exercise 2.
Complete the following cells to compute the variance and standard deviation of the incoming values using **custom** stateful processing (don't just use `processed_df.agg({"value" : "var"})`).

Sample variance sequential formulation:  $\sigma^2_{(N)} ={S_{(N)}/N}$ where $$S_{(N)}= S_{(N-1)}+N(N-1) (\mu_{(N)}- \mu_{(N-1)})^2 $$
and $\mu_{(N)}$ is the sample average computed after the $N^{th}$ sample

**Your Solution**

In [ ]:
# ...

In [ ]:
def update_state(batch_df, batch_id):
  ...

In [ ]:
# Create a streaming DataFrame
streaming_df = spark.readStream \
    .format("rate") \
    .option("rowsPerSecond", 10) \
    .load()

# Process the DataFrame
processed_df = streaming_df.select(
    col("value").alias("id"),
    (rand()*100).alias("value")  # generate random values
)


# Start the streaming query
query = processed_df.writeStream \
    .outputMode("append") \
    .format("memory") \
    .foreachBatch(update_state) \
    .start()

In [ ]:
# ...

In [ ]:
query.stop()

**Solution**

In [ ]:
conn = sqlite3.connect('/content/state.db')
cursor = conn.cursor()

cursor.execute('''DROP TABLE IF EXISTS state''')

In [ ]:
def update_state(batch_df, batch_id):
  if batch_df.count() > 0:
        # Create a new SQLite connection inside the function
        conn = sqlite3.connect('/content/state.db')
        cursor = conn.cursor()

        # Ensure state table exists
        cursor.execute('''
            CREATE TABLE IF NOT EXISTS state (
                total Real,
                S REAL,
                N INTEGER
            )
        ''')

        # Read previous state from SQLite
        cursor.execute('SELECT total, S, N FROM state')
        row = cursor.fetchone()
        if row:
            total, S, N = row
            avg = total / N
        else:
            total, S, N = 0, 0, 0  # Initialize if no state exists
            avg = 0


        for row in batch_df.collect():
          total += row.value
          N += 1
          newAvg = total/N
          S = S + N * (N - 1) * (newAvg - avg)**2
          avg = newAvg

        # Store updated state in SQLite
        cursor.execute('DELETE FROM state')  # Remove old state (keeping only one row)
        cursor.execute('INSERT INTO state (total, S, N) VALUES (?, ?, ?)', (total, S, N))
        conn.commit()

        # Close SQLite connection
        cursor.close()
        conn.close()

        display(f"Updated State -> total: {total}, count: {N}, var: {S/N}, std: {(S/N)**0.5}")


In [ ]:
# Create a streaming DataFrame
streaming_df = spark.readStream \
    .format("rate") \
    .option("rowsPerSecond", 10) \
    .load()

# Process the DataFrame
processed_df = streaming_df.select(
    col("value").alias("id"),
    (rand()*100).alias("value")  # generate random values
)


# Start the streaming query
query = processed_df.writeStream \
    .outputMode("append") \
    .format("memory") \
    .foreachBatch(update_state) \
    .start()

time.sleep(10)

'Updated State -> total: 438.6345772330578, count: 10, var: 488.21297325123487, std: 22.09554193160319'

'Updated State -> total: 1062.0167963637869, count: 20, var: 494.18524924726006, std: 22.230277759111786'

'Updated State -> total: 1517.1862147300167, count: 30, var: 640.7571470199762, std: 25.31318128999151'

'Updated State -> total: 1891.8034095868848, count: 40, var: 640.4060119901095, std: 25.306244525612833'

'Updated State -> total: 2422.083604768779, count: 50, var: 722.9278292025102, std: 26.887317255585582'

'Updated State -> total: 2952.72564258602, count: 60, var: 751.2749397360739, std: 27.409395099784195'

'Updated State -> total: 3692.223857619457, count: 70, var: 761.6446611519126, std: 27.597910449016112'

'Updated State -> total: 4311.16432647595, count: 80, var: 763.1679359384415, std: 27.625494311205394'

'Updated State -> total: 4795.801715306523, count: 90, var: 789.1030430560687, std: 28.09097796546195'

In [ ]:
cursor.execute('SELECT total, S, N FROM state')
row = cursor.fetchone()
avg = 0
if row:
    total, S, N = row
    var = S / N if N > 0 else 0
print(f"var : {var}, std: {var**0.5}, average: {total/N}")

var : 844.5909306520244, std: 29.06184664903496, average: 51.82127388584403


In [ ]:
query.stop()

'Updated State -> total: 21192.620666393985, count: 410, var: 859.0098835683186, std: 29.30887039052032'

## Recursive Least Squares (RLS)

The following equations define a single step of the Recursive Least Squares (with forgetting factor $\nu$).

\begin{equation*}
\begin{cases}
V_{(t)}&=\frac{1}{\nu} \left(V_{(t-1)}
-\frac{V_{(t-1)} x^T_{t} x_{t} V_{(t-1)}}{1+ x_{t} V_{(t-1)} x^T_{t}} \right)\\[3pt]
\alpha_{(t)}&= V_{(t)} x^T_{t} \\[3pt]
e&= y_{t}- x_{t} \hat{\beta}_{(t-1)}  \\[3pt]
\hat{\beta}_{(t)}&=\hat{\beta}_{(t-1)}+ \alpha_{(t)} e \\[3pt]
\end{cases}
\end{equation*}

where $V$ is the covariance matrix and $\beta$ is the set of parameters of the linear model.
Details in the Streaming Analytics slides.

- $\nu = 1$ : do not forget any sample
- $\nu \rightarrow 0$ : train only on the last samples


### Exercise 3.
Write the implementation of `RLS` in order to perform the update of the state of the RLS model according to the equations given above.

Show the Batch MSE using `display()` and show the fitted coefficients

**Your Solution**

In [ ]:
# ...

In [ ]:
def update_rls_state(batch_df, batch_id):
  ...

In [ ]:
# Create a streaming DataFrame
streaming_df = spark.readStream \
    .format("rate") \
    .option("rowsPerSecond", 5) \
    .load()

# Process the DataFrame
processed_df = streaming_df.select(
    col("value").alias("id"),
    (rand()*10).alias("x"),                         # generate random values
    (a * col("x") + b + rand() * 2 - 1).alias("y")  # y = ax + b + noise
)

# Start the streaming query
query = processed_df.writeStream \
    .format("memory") \
    .foreachBatch(update_rls_state) \
    .start()


time.sleep(10)

In [ ]:
# ...

In [ ]:
query.stop()

**Solution**

In [ ]:
conn = sqlite3.connect('/content/stateRLS.db')
cursor = conn.cursor()

cursor.execute('''
    DROP TABLE IF EXISTS stateRLS''')

In [ ]:
# Function to update state in SQLite
def update_rls_state(batch_df, batch_id):
    if batch_df.count() > 0:
        # Create a new SQLite connection inside the function
        conn = sqlite3.connect('/content/stateRLS.db')
        cursor = conn.cursor()

        # Ensure state table exists
        cursor.execute('''
            CREATE TABLE IF NOT EXISTS stateRLS (
                beta TEXT,
                V TEXT
            )
        ''')

        n = 1 # 1 feature
        nu = 1 # forgetting factor (0 = only train on last sample, 1 = don't forget any sample)

        # Read previous state from SQLite
        cursor.execute('SELECT beta, V FROM stateRLS')
        row = cursor.fetchone()
        if row:
            beta, V = row
            beta = np.array(ast.literal_eval(beta))
            V = np.array(ast.literal_eval(V))
        else:
            beta =  np.zeros(n+1) # for n features
            V = np.diag(np.zeros(n+1)+10)


        batchError = 0
        # Compute new state
        for row in batch_df.collect():
          x1 = [1]
          x1.append(row.x)
          x1 = np.array([x1])
          y = row.y

          V=1.0/nu*(V-V.dot(x1.T).dot(x1).dot(V)/(1.0+float(x1.dot(V).dot(x1.T))))
          alpha=V.dot(x1.T)
          yhat=x1.dot(beta)
          err=y-yhat
          beta=(beta.reshape(-1, 1) + alpha * err).flatten()

          batchError += (err**2)[0]
        display(f"Batch MSE : {batchError /  batch_df.count()}")


        new_beta = str(beta.tolist())
        new_V = str(V.tolist())

        # Store updated state in SQLite
        cursor.execute('DELETE FROM stateRLS')  # Remove old state (keeping only one row)
        cursor.execute('INSERT INTO stateRLS (beta, V) VALUES (?, ?)', (new_beta, new_V))
        conn.commit()

        # Close SQLite connection
        cursor.close()
        conn.close()

In [ ]:
# a*x + b = y
a = 6
b = 3

In [ ]:
# Create a streaming DataFrame
streaming_df = spark.readStream \
    .format("rate") \
    .option("rowsPerSecond", 5) \
    .load()

# Process the DataFrame
processed_df = streaming_df.select(
    col("value").alias("id"),
    (rand()*10).alias("x"),                         # generate random values
    (a * col("x") + b + rand() * 2 - 1).alias("y")  # y = ax + b + noise
)

# Start the streaming query
query = processed_df.writeStream \
    .format("memory") \
    .foreachBatch(update_rls_state) \
    .start()


time.sleep(10)

<ipython-input-114-11ca76b4deae>:39: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  V=1.0/nu*(V-V.dot(x1.T).dot(x1).dot(V)/(1.0+float(x1.dot(V).dot(x1.T))))


'Batch MSE : 279.61949765651013'

'Batch MSE : 0.32250534632750066'

'Batch MSE : 0.1357595721277838'

'Batch MSE : 0.1887067289946356'

'Batch MSE : 0.5413539415088491'

'Batch MSE : 0.2773719084334086'

'Batch MSE : 0.3372314389769526'

'Batch MSE : 0.07141308126849084'

'Batch MSE : 0.2481569101135134'

In [ ]:
cursor.execute('SELECT beta, V FROM stateRLS')
row = cursor.fetchone()
beta = 0
if row:
    beta, _ = row
    beta = np.array(ast.literal_eval(beta))
print(f"fitted regression : y = {beta[1]} x + {beta[0]}")

fitted regression : y = 5.995194872397716 x + 3.0662649409223164


In [ ]:
query.stop()

## Multiple RLS in parallel

### Exercise 4.
Use multiple queries to 2 concurent RLS in parallel with different forgetting factors

Compute and show the batch MSE and total MSE (sum of batch SE divided by the total number of samples) for each model and show the trained coefficients

**Your Solution**

In [ ]:
# ...

**Solution**

In [ ]:
conn = sqlite3.connect('/content/stateRLS.db')
cursor = conn.cursor()

cursor.execute('''DROP TABLE IF EXISTS stateRLS_0''')
cursor.execute('''DROP TABLE IF EXISTS stateRLS_1''')

In [ ]:


# Function to update state in SQLite
def update_rls_state(batch_df, batch_id, forgettingFactor=1, id=0):
    if batch_df.count() > 0:
        # Create a new SQLite connection inside the function
        conn = sqlite3.connect('/content/stateRLS.db')
        cursor = conn.cursor()

        # Ensure state table exists
        cursor.execute(f'''
            CREATE TABLE IF NOT EXISTS stateRLS_{id} (
                beta TEXT,
                V TEXT,
                totalSE REAL,
                countSample INTEGER
            )
        ''')

        n = 1 # 1 feature
        nu = forgettingFactor # forgetting factor (0 = only train on last sample, 1 = don't forget any sample)

        # Read previous state from SQLite
        cursor.execute(f'SELECT beta, V, totalSE, countSample FROM stateRLS_{id}')
        row = cursor.fetchone()
        if row:
            beta, V, tot, count = row
            beta = np.array(ast.literal_eval(beta))
            V = np.array(ast.literal_eval(V))
        else:
            beta =  np.zeros(n+1) # for n features
            V = np.diag(np.zeros(n+1)+10)
            tot = 0
            count = 0


        batchError = 0
        # Compute new state
        for row in batch_df.collect():
          x1 = [1]
          x1.append(row.x)
          x1 = np.array([x1])
          y = row.y

          V=1.0/nu*(V-V.dot(x1.T).dot(x1).dot(V)/(1.0+float(x1.dot(V).dot(x1.T))))
          alpha=V.dot(x1.T)
          yhat=x1.dot(beta)
          err=y-yhat
          beta=(beta.reshape(-1, 1) + alpha * err).flatten()

          batchError += (err**2)[0]

        tot += batchError
        count += batch_df.count()

        display(f"RLS {id} : Batch MSE : {batchError /  batch_df.count()}, total MSE : {tot / count}")


        new_beta = str(beta.tolist())
        new_V = str(V.tolist())

        # Store updated state in SQLite
        cursor.execute(f'DELETE FROM stateRLS_{id}')  # Remove old state (keeping only one row)
        cursor.execute(f'INSERT INTO stateRLS_{id} (beta, V, totalSE, countSample) VALUES (?, ?, ?, ?)', (new_beta, new_V, tot, count))
        conn.commit()

        # Close SQLite connection
        cursor.close()
        conn.close()

In [ ]:
# a*x + b = y
a = 6
b = 3

In [ ]:
# Create a streaming DataFrame
streaming_df = spark.readStream \
    .format("rate") \
    .option("rowsPerSecond", 5) \
    .load()

# Process the DataFrame
processed_df = streaming_df.select(
    col("value").alias("id"),
    (rand()*10).alias("x"),                         # generate random values
    (a * col("x") + b + rand() * 2 - 1).alias("y")  # y = ax + b + noise
)

query1 = processed_df.writeStream \
    .format("memory") \
    .foreachBatch(lambda batch_df, batch_id: update_rls_state(batch_df, batch_id, 1, id=0)) \
    .start()

query2 = processed_df.writeStream \
    .format("memory") \
    .foreachBatch(lambda batch_df, batch_id: update_rls_state(batch_df, batch_id, 0.95, id=1)) \
    .start()


time.sleep(10)

<ipython-input-120-f7f0d7391a58>:43: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  V=1.0/nu*(V-V.dot(x1.T).dot(x1).dot(V)/(1.0+float(x1.dot(V).dot(x1.T))))


'RLS 0 : Batch MSE : 755.1280218818299, total MSE : 755.1280218818299'

'RLS 1 : Batch MSE : 306.3694808767388, total MSE : 306.3694808767388'

'RLS 0 : Batch MSE : 0.11031567750000167, total MSE : 377.6191687796649'

'RLS 1 : Batch MSE : 0.43036279848312786, total MSE : 153.39992183761098'

...

'RLS 1 : Batch MSE : 0.5144662882049851, total MSE : 44.10941029083203'

'RLS 0 : Batch MSE : 0.5625016612015254, total MSE : 94.7412292459218'

'RLS 1 : Batch MSE : 0.4063364921771516, total MSE : 38.64652606600017'

'RLS 0 : Batch MSE : 0.29744046474474795, total MSE : 84.24747493690212'

'RLS 1 : Batch MSE : 0.21747061824673858, total MSE : 34.37663101624979'

In [ ]:
cursor.execute('SELECT beta, V FROM stateRLS_0')
row = cursor.fetchone()
beta = 0
if row:
    beta, _ = row
    beta = np.array(ast.literal_eval(beta))
print(f"RLS 0 : fitted regression : y = {beta[1]} x + {beta[0]}")

cursor.execute('SELECT beta, V FROM stateRLS_1')
row = cursor.fetchone()
beta = 0
if row:
    beta, _ = row
    beta = np.array(ast.literal_eval(beta))
print(f"RLS 1 : fitted regression : y = {beta[1]} x + {beta[0]}")

RLS 0 : fitted regression : y = 6.046913923805216 x + 2.9202034073562846
RLS 1 : fitted regression : y = 6.064999960440283 x + 2.6161813622869907


In [ ]:
query1.stop()
query2.stop()

## ML batch predictor (racing of several models)

To conclude this session, we implement a parallel racing of multiple machine learning predictors (here *Random Forests* - `rf`, *Linear Model* - `lin`, *Gradient boosting* - `gb`, respectively)

The aim is to use a predictor trained on the all previous batches to predict the labels of the new batch.

To this end, we store all the previous samples in the state and retrain, from scratch at every batch using `sklearn` models.

As we have 3 parallel workers, we only ask one of them to store the samples in the database. We also store the batch number with the samples to avoid training a model with the testing samples if another worker already sotred them in the database.

### Exercise 5.
Implement this racing of models

**Your Solution**

In [ ]:
# ...

In [ ]:
# ...

In [ ]:
# a*x² + bx + c = y
a = 5
b = 9
c = 2

In [ ]:
# Create a streaming DataFrame
streaming_df = spark.readStream \
    .format("rate") \
    .option("rowsPerSecond", 5) \
    .load()

# Process the DataFrame
processed_df = streaming_df.select(
    col("value").alias("id"),
    (rand()*10).alias("x1"),
    (rand()*10).alias("x2"),
    (a * col("x1")**2 + b * col("x2") + c + rand() * 2 - 1).alias("y")  # y = a*x² + bx + c = y + noise
)

# Start the streaming query
query1 = ...

query2 = ...

query3 = ...


time.sleep(10)

In [ ]:
query1.stop()
query2.stop()
query3.stop()

**Solution**

In [ ]:
conn = sqlite3.connect('/content/state_ML.db')
cursor = conn.cursor()

cursor.execute('''DROP TABLE IF EXISTS stateML''')

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

# Function to update state in SQLite
def update_ML_state(batch_df, batch_id, model, storeData=False):
    if batch_df.count() > 0:
        # Create a new SQLite connection inside the function
        conn = sqlite3.connect('/content/state_ML.db')
        cursor = conn.cursor()

        # Ensure state table exists
        cursor.execute(f'''
            CREATE TABLE IF NOT EXISTS stateML (
                batchId INTEGER,
                X1 REAL,
                X2 REAL,
                Y REAL
            )
        ''')


        # Read previous state from SQLite
        cursor.execute(f'SELECT X1, X2, Y FROM stateML WHERE batchId < {batch_id}')
        row = np.array(cursor.fetchall())
        if len(row) > 0:
            x = row[:, :2]
            y = row[:, 2]
        else:
            x = np.array([])
            y = np.array([])

        x1_new = np.array(batch_df.select("x1").collect())
        x2_new = np.array(batch_df.select("x2").collect())
        x_new = np.concatenate((x1_new, x2_new), axis=1)
        y_new = np.array(batch_df.select("y").collect())


        if len(x) > 0:
          regr = None
          if model=="rf":
              regr = RandomForestRegressor(random_state=0)
          if model=="gb":
              regr = GradientBoostingRegressor(random_state=0)
          if model=="lin":
              regr=LinearRegression()
          regr.fit(x, y)

          pred = regr.predict(x_new)
          mse = mean_squared_error(y_new, pred)


          display(f"{model} : Batch MSE : {mse}")
        else:
          display(f"{model} : No data")

        # Store updated state in SQLite
        if storeData:
          for i in range(len(x1_new)):
            cursor.execute(f'INSERT INTO stateML (batchId, X1, X2, Y) VALUES (?, ?, ?, ?)', (batch_id, x1_new[i][0], x2_new[i][0], y_new[i][0]))
          conn.commit()

        # Close SQLite connection
        cursor.close()
        conn.close()

In [ ]:
# a*x² + bx + c = y
a = 5
b = 9
c = 2

In [ ]:
# Create a streaming DataFrame
streaming_df = spark.readStream \
    .format("rate") \
    .option("rowsPerSecond", 5) \
    .load()

# Process the DataFrame
processed_df = streaming_df.select(
    col("value").alias("id"),
    (rand()*10).alias("x1"),
    (rand()*10).alias("x2"),
    (a * col("x1")**2 + b * col("x2") + c + rand() * 2 - 1).alias("y")  # y = a*x² + bx + c = y + noise
)

# Start the streaming query
query1 = processed_df.writeStream \
    .format("memory") \
    .foreachBatch(lambda batch_df, batch_id: update_ML_state(batch_df, batch_id, "rf", storeData=True)) \
    .start()

query2 = processed_df.writeStream \
    .format("memory") \
    .foreachBatch(lambda batch_df, batch_id: update_ML_state(batch_df, batch_id, "gb")) \
    .start()

query3 = processed_df.writeStream \
    .format("memory") \
    .foreachBatch(lambda batch_df, batch_id: update_ML_state(batch_df, batch_id, "lin")) \
    .start()


time.sleep(10)

'rf : No data'

'gb : No data'

'lin : No data'

'rf : Batch MSE : 12592.354195322238'

'gb : Batch MSE : 1425.1053063946758'

...

'gb : Batch MSE : 725.1113665797627'

'lin : Batch MSE : 1473.4260327581271'

'lin : Batch MSE : 601.3517346753104'

'gb : Batch MSE : 203.54580756603633'

'rf : Batch MSE : 581.4003386243818'

In [ ]:
query1.stop()
query2.stop()
query3.stop()

## Finally

in the end, stop the spark session

In [ ]:
spark.stop()